In [1]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()

In [2]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [3]:
# Widget definitions (on Databricks) / config (Databricks Connect - widgets API limited)
_scope = "epic_on_fhir_oauth_keys"
_client_key = "client_id"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_algo = "RS384"

try:
    dbutils.widgets.text("epic_secrets_scope", _scope, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("algo", _algo, "Epic Token Enrcyption Algorithm")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

Box(children=(Label(value='Epic Secrets Scope'), Text(value='epic_on_fhir_oauth_keys')))

Box(children=(Label(value='Epic Client ID DB Secrets Key'), Text(value='client_id')))

Box(children=(Label(value='Epic Token URL'), Text(value='https://fhir.epic.com/interconnect-fhir-oauth/oauth2/…

Box(children=(Label(value='Epic Token Enrcyption Algorithm'), Text(value='RS384')))

In [4]:
# Databricks Connect doesn't support widgets.getAll(); use config fallback for local dev
try:
    widget_values = dbutils.widgets.getAll()
except AttributeError:
    widget_values = {
        "epic_secrets_scope": _scope,
        "client_id_dbs_key": _client_key,
        "token_url": _token_url,
        "algo": _algo,
    }

for name, value in widget_values.items():
    print(f"{name} = {value}")

epic_secrets_scope = epic_on_fhir_oauth_keys
client_id_dbs_key = client_id
token_url = https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token
algo = RS384


In [5]:
CLIENT_ID = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = widget_values["client_id_dbs_key"])
TOKEN_URL = widget_values["token_url"]
PRIVATE_KEY = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = "private_key")
ALGO = widget_values["algo"]
EPIC_KID = dbutils.secrets.get(scope=widget_values["epic_secrets_scope"], key="kid")

In [6]:
# print(f"""
# CLIENT_ID = {CLIENT_ID}
# TOKEN_URL = {TOKEN_URL}
# PRIVATE_KEY = {PRIVATE_KEY}
# ALGO = {ALGO}
# EPIC_KID = {EPIC_KID}
# """)

In [7]:
import sys
import os

# Get the directory containing this notebook
notebook_dir = os.path.dirname(os.path.abspath('__file__'))

# Add src directory to path if not already present (notebook is already in src/)
if notebook_dir not in sys.path:
	sys.path.append(notebook_dir)

from epic_on_fhir.auth import EpicApiAuth
from epic_on_fhir.endpoint import EpicApiRequest

In [8]:
epicAuth = EpicApiAuth(
  client_id = CLIENT_ID,
  private_key = PRIVATE_KEY,
  kid = EPIC_KID,
  algo = ALGO,
  auth_location = TOKEN_URL
)

In [9]:
epicAuth.can_connect()

True

In [10]:
epicFhirAPi = EpicApiRequest(
    auth = epicAuth,
    base_url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/"
)

In [11]:
resource = "Patient"
action = "?identifier=EXTERNAL|Z6129"

In [12]:
patient = epicFhirAPi.make_request(
    http_method="get",
    resource=resource,
    action=action
)

In [13]:
import json

# Display the full response structure
print("Response keys:", patient.keys())
print("\nResponse status:", patient['response']['response_status_code'])
print("\nResponse text (first 500 chars):")
print(patient['response']['response_text'][:500])

Response keys: dict_keys(['request', 'response'])

Response status: 200

Response text (first 500 chars):
{"resourceType":"Bundle","type":"searchset","total":1,"link":[{"relation":"self","url":"https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient?identifier=EXTERNAL|Z6129"}],"entry":[{"link":[{"relation":"self","url":"https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/erXuFYUfucBZaryVksYEcMg3"}],"fullUrl":"https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/erXuFYUfucBZaryVksYEcMg3","resource":{"resourceType":"Patient","id":"erXuFYUfucBZaryVksYEcMg3","extens


In [14]:
# Parse the response text as JSON
patient_data = json.loads(patient['response']['response_text'])

# Display the parsed patient data
display(patient_data)

{'resourceType': 'Bundle',
 'type': 'searchset',
 'total': 1,
 'link': [{'relation': 'self',
   'url': 'https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient?identifier=EXTERNAL|Z6129'}],
 'entry': [{'link': [{'relation': 'self',
     'url': 'https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/erXuFYUfucBZaryVksYEcMg3'}],
   'fullUrl': 'https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/erXuFYUfucBZaryVksYEcMg3',
   'resource': {'resourceType': 'Patient',
    'id': 'erXuFYUfucBZaryVksYEcMg3',
    'extension': [{'valueCodeableConcept': {'coding': [{'system': 'urn:oid:1.2.840.114350.1.13.0.1.7.10.698084.130.657370.19999000',
         'code': 'female',
         'display': 'female'}],
       'text': 'Female'},
      'url': 'http://open.epic.com/FHIR/StructureDefinition/extension/legal-sex'},
     {'valueCodeableConcept': {'coding': [{'system': 'urn:oid:1.2.840.114350.1.13.0.1.7.10.698084.130.657370.19999000',
         'code': 'female',
         'dis

In [15]:
# Extract the FHIR identifier from the patient resource
identifiers = patient_data['entry'][0]['resource']['identifier']
patient_id = None

for identifier in identifiers:
	if identifier.get('type', {}).get('text') == 'FHIR':
		patient_id = identifier['value']
		break

print(f"Patient FHIR ID: {patient_id}")

Patient FHIR ID: TnOZ.elPXC6zcBNFMcFA7A5KZbYxo2.4T-LylRk4GoW4B


In [16]:
patient_id = "erXuFYUfucBZaryVksYEcMg3"

In [17]:
resource = "Patient"
action = f"{patient_id}/$summary"
print(action)

erXuFYUfucBZaryVksYEcMg3/$summary


In [18]:
epicFhirAPi.make_request(
    http_method="get",
    resource=resource,
    action=action
)

{'request': {'http_method': 'get',
  'url': 'https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/erXuFYUfucBZaryVksYEcMg3/$summary',
  'data': ''},
 'response': {'response_status_code': 200,
  'response_time_seconds': 0.340253,
  'response_headers': {'Cache-Control': 'no-cache,no-store', 'Pragma': 'no-cache', 'Content-Length': '4718', 'Content-Type': 'application/json; charset=utf-8', 'Content-Encoding': 'gzip', 'Expires': '-1', 'ServerMetrics': '{"BlockReads":1453,"BlockWrites":10,"BlocksAllocated":9,"DBTime":841,"DatabaseCPUTime":144,"ECFNetworkTime":41,"ECFRequestCount":1,"ECFRequestTime":882,"GREF":15544,"JournalEntries":4,"LockWait":0.045,"LocksFailed":0,"LocksGranted":0,"MCommands":357670,"MemoryDifference":0,"NetworkCacheMisses":0,"NetworkUpdates":0,"WorkflowEventBlockReads":1453,"WorkflowEventDBTime":841,"WorkflowEventECFNetworkTime":40,"WorkflowEventECFRequestCount":1,"WorkflowEventGREF":15544}', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains',